#### Noisy Quantum SVM - Lung Cancer Dataset

**Complete parameter testing:** features (qubits), shots, reps, entanglement, noise levels

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# Reproducibility
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, recall_score
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

In [ ]:
# Qiskit imports
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

#### Data Loading

In [ ]:
# Load lung cancer data
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path_lung = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'

df_lung = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])
print(f"Original shape: {df_lung.shape}")

#### Data Preprocessing

In [ ]:
# Mode imputation for missing values
modes = df_lung.mode().iloc[0]
df_lung.fillna(modes, inplace=True)
print(f"Missing values after imputation: {df_lung.isnull().sum().sum()}")

# Binary label encoding (0 = class 1, 1 = others)
df_lung['label_binary'] = df_lung['label'].apply(lambda x: 0 if x == 1 else 1)

# Features and labels
X_lung = df_lung.drop(['label', 'label_binary'], axis=1)
y_lung_binary = df_lung['label_binary']

# Train-test split (70-30)
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lung, y_lung_binary, test_size=0.3, random_state=42, stratify=y_lung_binary
)
print(f"Train: {X_train_lc.shape[0]}, Test: {X_test_lc.shape[0]}")

In [ ]:
# One-hot encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_lc_encoded = pd.DataFrame(encoder.fit_transform(X_train_lc), columns=encoder.get_feature_names_out())
X_test_lc_encoded = pd.DataFrame(encoder.transform(X_test_lc), columns=encoder.get_feature_names_out())
print(f"After encoding: {X_train_lc_encoded.shape[1]} features")

#### Feature Selection (Cramér's V)

In [ ]:
# Cramér's V correlation for categorical features
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0: 
        return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# Compute feature importance scores
cramers_scores = {col: cramers_v(X_train_lc_encoded[col], y_train_lc) for col in X_train_lc_encoded.columns}
cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)

print("Feature importance scores computed!")
print(f"Total encoded features: {len(cramers_series)}")
print(f"\nTop 10 features:")
for i, (feat, score) in enumerate(cramers_series.head(10).items(), 1):
    print(f"  {i}. {feat}: {score:.4f}")

#### Noise Model Factory

In [ ]:
def get_noise_model(level='standard'):
    """
    Returns noise model, backend, pass manager for given noise level.

    Parameters:
    - 'low': 0.01% 1q, 0.1% 2q, 0.2% readout
    - 'standard': 0.1% 1q, 1.0% 2q, 2.0% readout (realistic NISQ)
    - 'high': 0.5% 1q, 5.0% 2q, 10.0% readout
    """
    noise_params = {
        'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
        'standard': {'p_1q': 0.001, 'p_2q': 0.01, 'p_ro': 0.02},
        'high': {'p_1q': 0.005, 'p_2q': 0.05, 'p_ro': 0.10},
    }

    params = noise_params.get(level, noise_params['standard'])

    # Build noise model
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(params['p_1q'], 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(params['p_2q'], 2), ['cx'])
    readout_error = ReadoutError([[1 - params['p_ro'], params['p_ro']], [params['p_ro'], 1 - params['p_ro']]])
    noise_model.add_all_qubit_readout_error(readout_error)

    # Backend with noise
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

    return noise_model, backend, pm, params

print("Noise model factory ready!")
print("Levels: 'low', 'standard', 'high'")

#### Experiment Configurations

In [ ]:
# All experiments (17 total)
experiments = [
    # --- EXP 2: Dimensionality Effect (Qubits/Features) ---
    {'id': 'Exp2_2feat',  'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_4feat',  'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_6feat',  'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_8feat',  'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat', 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat', 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_10feat', 'k_features': 15, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 3: Shot Noise Effect ---
    {'id': 'Exp3_128shots',  'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_512shots',  'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_1024shots', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 4: Circuit Depth (Reps) ---
    {'id': 'Exp4_Reps1', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps2', 'k_features': 8, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp4_Reps3', 'k_features': 8, 'shots': 1024, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 5: Entanglement Topology ---
    {'id': 'Exp5_Linear',   'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear',   'noise_level': 'standard'},
    {'id': 'Exp5_Circular', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp5_Full',     'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard'},

    # --- EXP 6: Noise Level Ablation ---
    {'id': 'Exp6_LowNoise',  'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    {'id': 'Exp6_StdNoise',  'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp6_HighNoise', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments: {len(experiments)}")
print(f"Dataset: {X_train_lc.shape[0]} train + {X_test_lc.shape[0]} test = {len(df_lung)} samples")
print("\nParameters tested: k_features, shots, reps, entanglement, noise")

#### Main Experiment Loop

In [ ]:
# Prepare numpy arrays
X_train_encoded_np = np.asarray(X_train_lc_encoded)
X_test_encoded_np  = np.asarray(X_test_lc_encoded)
y_train_np = np.asarray(y_train_lc)
y_test_np  = np.asarray(y_test_lc)

# Determine CV splits
class_counts = np.bincount(y_train_np.astype(int))
min_class = int(class_counts.min()) if len(class_counts) > 0 else 1
n_splits = max(2, min(3, min_class))

print(f"CV: {n_splits}-fold StratifiedKFold (min class size={min_class})")
print("\nStarting experiments...\n")

all_results = []

for exp_num, config in enumerate(experiments, 1):
    print("\n" + "="*80)
    print(f"EXPERIMENT {exp_num}/{len(experiments)}: {config['id']}")
    print("="*80)
    print(f" K Features (Qubits): {config['k_features']}")
    print(f" Shots: {config['shots']}")
    print(f" Reps: {config['reps']}")
    print(f" Entanglement: {config['entanglement']}")
    print(f" Noise Level: {config.get('noise_level', 'standard')}")

    # ===================================================================
    # DYNAMIC FEATURE SELECTION
    # ===================================================================
    k_features = config['k_features']
    selected_features = cramers_series.head(k_features).index.tolist()
    feature_indices = [X_train_lc_encoded.columns.get_loc(feat) for feat in selected_features]

    X_train_k = X_train_encoded_np[:, feature_indices]
    X_test_k  = X_test_encoded_np[:, feature_indices]

    print(f" Selected: {selected_features[:3]}... ({k_features} total)")

    # ===================================================================
    # NOISE MODEL
    # ===================================================================
    noise_level = config.get('noise_level', 'standard')
    noise_model, backend, pm, noise_params = get_noise_model(noise_level)
    print(f" Noise: 1q={noise_params['p_1q']*100:.3f}%, 2q={noise_params['p_2q']*100:.2f}%, RO={noise_params['p_ro']*100:.1f}%")

    # ===================================================================
    # QUANTUM KERNEL
    # ===================================================================
    sampler = AerSampler.from_backend(backend=backend, default_shots=int(config['shots']))
    fm = ZZFeatureMap(feature_dimension=k_features, reps=int(config['reps']), entanglement=config['entanglement'])
    fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
    qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)

    # ===================================================================
    # KERNEL MATRICES
    # ===================================================================
    print(" Computing kernels...")
    t0 = time.time()
    K_train = qkernel.evaluate(x_vec=X_train_k)
    K_test = qkernel.evaluate(x_vec=X_test_k, y_vec=X_train_k)
    kernel_time = time.time() - t0
    print(f" Kernel time: {kernel_time:.2f}s")

    # ===================================================================
    # GRID SEARCH
    # ===================================================================
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    grid = GridSearchCV(
        SVC(kernel='precomputed', class_weight='balanced'),
        param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
    )

    t1 = time.time()
    grid.fit(K_train, y_train_np)
    train_time = time.time() - t1

    best_model = grid.best_estimator_
    best_c = grid.best_params_['C']
    cv_score = grid.best_score_

    # ===================================================================
    # EVALUATION
    # ===================================================================
    y_train_pred = best_model.predict(K_train)
    y_test_pred  = best_model.predict(K_test)

    train_acc = accuracy_score(y_train_np, y_train_pred)
    test_acc  = accuracy_score(y_test_np, y_test_pred)
    test_bal_acc = balanced_accuracy_score(y_test_np, y_test_pred)
    recall_pos1 = recall_score(y_test_np, y_test_pred, pos_label=1, zero_division=0)
    gen_gap = abs(train_acc - test_acc)

    print(f" C={best_c} | CV={cv_score:.4f} | Train time={train_time:.2f}s")
    print(f" Train={train_acc:.4f} | Test={test_acc:.4f} | Balanced={test_bal_acc:.4f} | Recall(1)={recall_pos1:.4f} | Gap={gen_gap:.4f}")

    print("\n Classification Report:")
    print(classification_report(y_test_np, y_test_pred, zero_division=0))

    # ===================================================================
    # STORE RESULTS
    # ===================================================================
    all_results.append({
        'experiment_id': config['id'],
        'exp_number': exp_num,
        'total_samples': int(X_train_k.shape[0] + X_test_k.shape[0]),
        'train_samples': int(X_train_k.shape[0]),
        'test_samples': int(X_test_k.shape[0]),
        'k_features': int(k_features),
        'shots': int(config['shots']),
        'reps': int(config['reps']),
        'entanglement': config['entanglement'],
        'noise_level': noise_level,
        'p_1q': noise_params['p_1q'],
        'p_2q': noise_params['p_2q'],
        'p_ro': noise_params['p_ro'],
        'selected_features': selected_features,
        'best_c': float(best_c),
        'cv_score': float(cv_score),
        'train_acc': float(train_acc),
        'test_acc': float(test_acc),
        'test_balanced_acc': float(test_bal_acc),
        'recall_pos1': float(recall_pos1),
        'gen_gap': float(gen_gap),
        'kernel_time': float(kernel_time),
        'train_time': float(train_time),
    })

    # Save kernel matrices
    np.save(f'kernel_train_{config["id"]}.npy', K_train)
    np.save(f'kernel_test_{config["id"]}.npy', K_test)

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE!")
print("="*80)

#### Results Analysis

In [ ]:
# Results dataframe
results_df = pd.DataFrame(all_results)
print("\nResults Summary:")
print(results_df[['experiment_id','k_features','shots','reps','entanglement','noise_level','test_acc','test_balanced_acc','recall_pos1','gen_gap']].to_string(index=False))

out_csv = 'noisy_qsvm_lungcancer_results.csv'
results_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# Find best configurations
print("\n" + "=" * 80)
print("BEST CONFIGURATIONS")
print("=" * 80)

# Best test accuracy
best_acc_idx = results_df['test_acc'].idxmax()
best_acc = results_df.iloc[best_acc_idx]

print("\n★ BEST TEST ACCURACY:")
print(f"  {best_acc['experiment_id']}")
print(f"  Accuracy: {best_acc['test_acc']:.4f} | Recall(1): {best_acc['recall_pos1']:.4f} | Gap: {best_acc['gen_gap']:.4f}")
print(f"  Config: k={best_acc['k_features']}, shots={best_acc['shots']}, reps={best_acc['reps']}, ent={best_acc['entanglement']}, noise={best_acc['noise_level']}")

# Best recall
best_rec_idx = results_df['recall_pos1'].idxmax()
best_rec = results_df.iloc[best_rec_idx]

print("\n★ BEST RECALL (Label=1):")
print(f"  {best_rec['experiment_id']}")
print(f"  Accuracy: {best_rec['test_acc']:.4f} | Recall(1): {best_rec['recall_pos1']:.4f} | Gap: {best_rec['gen_gap']:.4f}")
print(f"  Config: k={best_rec['k_features']}, shots={best_rec['shots']}, reps={best_rec['reps']}, ent={best_rec['entanglement']}, noise={best_rec['noise_level']}")

# Best generalization
best_gen_idx = results_df['gen_gap'].idxmin()
best_gen = results_df.iloc[best_gen_idx]

print("\n★ BEST GENERALIZATION:")
print(f"  {best_gen['experiment_id']}")
print(f"  Accuracy: {best_gen['test_acc']:.4f} | Recall(1): {best_gen['recall_pos1']:.4f} | Gap: {best_gen['gen_gap']:.4f}")
print(f"  Config: k={best_gen['k_features']}, shots={best_gen['shots']}, reps={best_gen['reps']}, ent={best_gen['entanglement']}, noise={best_gen['noise_level']}")

print("=" * 80)